In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "WHQ_G.xpt"
csv_file = DATA_DIR / "WHQ_G_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 80)
print("NHANES 2011-2012 WHQ_G XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WHD010",
    "WHD020",
    "WHQ030",
    "WHQ040",
    "WHD045",
    "WHD050",
    "WHQ060",
    "WHQ070",
    "WHD080A",
    "WHD080B",
    "WHD080C",
    "WHD080D",
    "WHD080E",
    "WHD080F",
    "WHD080G",
    "WHD080H",
    "WHD080I",
    "WHD080J",
    "WHD080K",
    "WHD080M",
    "WHD080N",
    "WHD080O",
    "WHD080P",
    "WHD080Q",
    "WHD080R",
    "WHD080S",
    "WHD080T",
    "WHD080L",
    "WHQ270",
    "WHQ280A",
    "WHQ280B",
    "WHQ280C",
    "WHQ280D",
    "WHQ280E",
    "WHQ090",
    "WHD100A",
    "WHD100B",
    "WHD100C",
    "WHD100D",
    "WHD100E",
    "WHD100F",
    "WHD100G",
    "WHD100H",
    "WHD100I",
    "WHD100J",
    "WHD100K",
    "WHD100M",
    "WHD100N",
    "WHD100O",
    "WHD100P",
    "WHD100Q",
    "WHD100R",
    "WHD100S",
    "WHD100T",
    "WHD100L",
    "WHQ210",
    "WHD220",
    "WHD110",
    "WHD120",
    "WHD130",
    "WHD140",
    "WHQ150"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "WHQ_G column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 6175:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 63:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,175 rows x 63 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# NEVER globally replace this without validating
# the exact variable and count first.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)


print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print("\nTiny values by variable:")


print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 5. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact attached WHQ_G.xpt pattern:
#
#     WHD220 = 97
#
# TOTAL = 97
#
#
# CDC:
#
# WHD220 =
#     Weight loss most successful (pounds)
#
# Valid released range:
#
#     0-250
#
# Therefore zero is a legitimate released value.
#

EXPECTED_TINY_COUNTS = {
    "WHD220": 97
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated WHQ_G pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 97:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary zeros existed "
        "before correction."
    )


print(
    "\nPASS: Exact WHD220 97-value "
    "tiny-zero pattern validated."
)


# ============================================================
# 6. RESTORE ONLY VALIDATED WHD220 ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#

whd220_tiny_mask = (
    df["WHD220"]
    .eq(TINY_VALUE)
)


correction_count = int(
    whd220_tiny_mask.sum()
)


if correction_count != 97:

    raise ValueError(
        "Unexpected WHD220 tiny-value count."
    )


df.loc[
    whd220_tiny_mask,
    "WHD220"
] = 0


print("\n--- VALIDATED ZERO RESTORATION ---")


print(
    "WHD220 restored to zero:",
    correction_count
)


if int(
    df["WHD220"]
    .eq(0)
    .sum()
) != 97:

    raise ValueError(
        "WHD220 zero restoration failed."
    )


print(
    "PASS: 97 validated WHD220 zeros restored."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# numeric source should contain.
#
# ONLY:
#
#     WHD220 tiny -> 0
#
# is permitted to change.
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["WHD220"]
    .eq(TINY_VALUE),
    "WHD220"
] = 0


# ============================================================
# 8. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining after correction:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 9. CHECK FOR GENUINE DECIMAL VARIABLES
# ============================================================

#
# CRITICAL:
#
# Run this BEFORE any Int64 conversion.
#
# Attached WHQ_G contains:
#
#     genuine fractional variables = NONE
#
# Even though several variables are heights/weights,
# the released WHQ_G values in this XPT are stored
# as whole-number inches/pounds.
#
# If a different/local file ever contains genuine
# fractions, STOP rather than rounding them.
#

print("\n--- GENUINE DECIMAL CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


        print(
            f"{col}: "
            f"{fractional_count:,} "
            "fractional observations"
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values unexpectedly "
        "found in WHQ_G. They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 10. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 62161
    and
    df["SEQN"].max() == 71916
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 11. FORMAT ALL PROVEN WHOLE-NUMBER VARIABLES
# ============================================================

#
# Attached WHQ_G contains no genuine decimal variables.
#
# Therefore all 63 fields can safely become
# pandas nullable Int64.
#
# IMPORTANT:
#
# NO .round() is used for conversion.
#

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely become Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 63 WHQ_G fields converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 12. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 13. CDC CHECK - WHD010
# ============================================================

#
# Current self-reported height in inches.
#
# CDC:
#
# 53-84 = 6060
# 7777  = 2
# 9999  = 82
# Missing = 31
#

s = df["WHD010"]


if not (
    int(
        s.between(
            53,
            84
        ).sum()
    ) == 6060

    and int(
        s.eq(7777).sum()
    ) == 2

    and int(
        s.eq(9999).sum()
    ) == 82

    and int(
        s.isna().sum()
    ) == 31
):

    raise ValueError(
        "WHD010 differs from CDC."
    )


print(
    "PASS: WHD010 matches CDC."
)


# ============================================================
# 14. CDC CHECK - WHD020
# ============================================================

#
# Current self-reported weight in pounds.
#
# CDC:
#
# 70-464 = 6074
# 7777   = 10
# 9999   = 57
# Missing = 34
#

s = df["WHD020"]


if not (
    int(
        s.between(
            70,
            464
        ).sum()
    ) == 6074

    and int(
        s.eq(7777).sum()
    ) == 10

    and int(
        s.eq(9999).sum()
    ) == 57

    and int(
        s.isna().sum()
    ) == 34
):

    raise ValueError(
        "WHD020 differs from CDC."
    )


print(
    "PASS: WHD020 matches CDC."
)


# ============================================================
# 15. CDC CHECK - WHQ030
# ============================================================

check_frequency(
    "WHQ030",
    {
        1: 2902,
        2: 374,
        3: 2878,
        7: 6,
        9: 14
    },
    1
)


# ============================================================
# 16. CDC CHECK - WHQ040
# ============================================================

check_frequency(
    "WHQ040",
    {
        1: 561,
        2: 3481,
        3: 2121,
        7: 3,
        9: 8
    },
    1
)


# ============================================================
# 17. CDC CHECK - WHD045
# ============================================================

#
# Desired weight in pounds.
#
# CDC:
#
# 85-320 = 4006
# 77777  = 4
# 99999  = 21
# Missing = 2144
#

s = df["WHD045"]


if not (
    int(
        s.between(
            85,
            320
        ).sum()
    ) == 4006

    and int(
        s.eq(77777).sum()
    ) == 4

    and int(
        s.eq(99999).sum()
    ) == 21

    and int(
        s.isna().sum()
    ) == 2144
):

    raise ValueError(
        "WHD045 differs from CDC."
    )


print(
    "PASS: WHD045 matches CDC."
)


# ============================================================
# 18. CDC CHECK - WHD050
# ============================================================

#
# Self-reported weight one year ago.
#

s = df["WHD050"]


if not (
    int(
        s.between(
            70,
            485
        ).sum()
    ) == 6020

    and int(
        s.eq(7777).sum()
    ) == 14

    and int(
        s.eq(9999).sum()
    ) == 109

    and int(
        s.isna().sum()
    ) == 32
):

    raise ValueError(
        "WHD050 differs from CDC."
    )


print(
    "PASS: WHD050 matches CDC."
)


# ============================================================
# 19. CDC CHECK - WHQ060
# ============================================================

check_frequency(
    "WHQ060",
    {
        1: 712,
        2: 422
    },
    5041
)


# ============================================================
# 20. CDC CHECK - WHQ070
# ============================================================

check_frequency(
    "WHQ070",
    {
        1: 1789,
        2: 3658,
        7: 4,
        9: 7
    },
    717
)


# ============================================================
# 21. CDC CHECK - WHQ090
# ============================================================

check_frequency(
    "WHQ090",
    {
        1: 613,
        2: 3048,
        7: 3,
        9: 5
    },
    2506
)


# ============================================================
# 22. CDC CHECK - WHQ210
# ============================================================

check_frequency(
    "WHQ210",
    {
        1: 1009,
        2: 2652,
        7: 3,
        9: 5
    },
    2506
)


# ============================================================
# 23. CDC CHECK - WHD220
# ============================================================

#
# Weight loss most successful in pounds.
#
# CDC:
#
# 0-250   = 3412
# 77777   = 4
# 99999   = 67
# Missing = 2692
#
#
# Attached pandas-decoded XPT originally contains
# 97 tiny values in this variable.
#
# After correction:
#
#     WHD220=0 -> exactly 97
#

s = df["WHD220"]


print("\n--- WHD220 ---")


print(
    "Restored zeros:",
    int(
        s.eq(0)
        .sum()
    )
)


print(
    "Values 0-250:",
    int(
        s.between(
            0,
            250
        ).sum()
    )
)


if not (
    int(
        s.between(
            0,
            250
        ).sum()
    ) == 3412

    and int(
        s.between(
            1,
            250
        ).sum()
    ) == 3315

    and int(
        s.eq(0).sum()
    ) == 97

    and int(
        s.eq(77777).sum()
    ) == 4

    and int(
        s.eq(99999).sum()
    ) == 67

    and int(
        s.isna().sum()
    ) == 2692
):

    raise ValueError(
        "WHD220 differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: WHD220 matches CDC including "
    "97 restored zeros."
)


# ============================================================
# 24. CDC CHECK - WHD110
# ============================================================

#
# Self-reported weight 10 years ago.
#

s = df["WHD110"]


if not (
    int(
        s.between(
            70,
            550
        ).sum()
    ) == 3795

    and int(
        s.eq(7777).sum()
    ) == 7

    and int(
        s.eq(9999).sum()
    ) == 143

    and int(
        s.isna().sum()
    ) == 2230
):

    raise ValueError(
        "WHD110 differs from CDC."
    )


print(
    "PASS: WHD110 matches CDC."
)


# ============================================================
# 25. CDC CHECK - WHD120
# ============================================================

#
# Self-reported weight at age 25.
#

s = df["WHD120"]


if not (
    int(
        s.between(
            80,
            413
        ).sum()
    ) == 4507

    and int(
        s.eq(7777).sum()
    ) == 9

    and int(
        s.eq(9999).sum()
    ) == 273

    and int(
        s.isna().sum()
    ) == 1386
):

    raise ValueError(
        "WHD120 differs from CDC."
    )


print(
    "PASS: WHD120 matches CDC."
)


# ============================================================
# 26. CDC CHECK - WHD130
# ============================================================

#
# Self-reported height at age 25 in inches.
#

s = df["WHD130"]


if not (
    int(
        s.between(
            49,
            83
        ).sum()
    ) == 2552

    and int(
        s.eq(7777).sum()
    ) == 4

    and int(
        s.eq(9999).sum()
    ) == 133

    and int(
        s.isna().sum()
    ) == 3486
):

    raise ValueError(
        "WHD130 differs from CDC."
    )


print(
    "PASS: WHD130 matches CDC."
)


# ============================================================
# 27. CDC CHECK - WHD140
# ============================================================

#
# Greatest self-reported weight in pounds.
#

s = df["WHD140"]


if not (
    int(
        s.between(
            75,
            550
        ).sum()
    ) == 5744

    and int(
        s.eq(7777).sum()
    ) == 17

    and int(
        s.eq(9999).sum()
    ) == 83

    and int(
        s.isna().sum()
    ) == 331
):

    raise ValueError(
        "WHD140 differs from CDC."
    )


print(
    "PASS: WHD140 matches CDC."
)


# ============================================================
# 28. CDC CHECK - WHQ150
# ============================================================

#
# Age when at heaviest weight.
#
# CDC:
#
# 7-79 = 5614
# 80   = 85
# 99999 = 35
# Missing = 441
#
# Age >=80 was top-coded to 80.
#

s = df["WHQ150"]


if not (
    int(
        s.between(
            7,
            79
        ).sum()
    ) == 5614

    and int(
        s.eq(80).sum()
    ) == 85

    and int(
        s.eq(77777).sum()
    ) == 0

    and int(
        s.eq(99999).sum()
    ) == 35

    and int(
        s.isna().sum()
    ) == 441
):

    raise ValueError(
        "WHQ150 differs from CDC."
    )


print(
    "PASS: WHQ150 matches CDC."
)


# ============================================================
# 29. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Raw XPT contained no ordinary numeric zeros.
#
# Therefore after correction the ONLY zeros in
# the complete WHQ_G file should be:
#
#     WHD220 = 97
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO PATTERN ---")


print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs from "
        "validated WHQ_G pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {final_zero_counts}"
    )


print(
    "PASS: Final zero pattern exactly matches "
    "the validated WHD220 correction."
)


# ============================================================
# 30. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY allowed numeric difference from the
# original pandas-decoded XPT:
#
#     WHD220:
#     97 TINY_VALUE -> 0
#
# Nothing else may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 63 fields match the expected "
    "corrected XPT exactly."
)


# ============================================================
# 31. EXPORT CSV
# ============================================================

#
# CRITICAL STANDARD RULE:
#
# NO:
#
#     .round()
#
# NO:
#
#     float_format=
#
# Although WHQ_G contains no genuine decimals,
# retaining the same export rule protects the overall
# NHANES workflow and avoids accidentally applying
# formatting rules to Dietary/Lab decimal files.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 32. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 33. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 63 CSV fields round-trip "
    "to exact corrected XPT values."
)


# ============================================================
# 34. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 35. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 36. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    numeric_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        numeric_values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 37. FINAL CSV ZERO-PATTERN CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    numeric_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        numeric_values
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO PATTERN ---")


print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "97 WHD220 zeros and no other zeros."
)


# ============================================================
# 38. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# WHQ_G contains no genuine decimal fields.
#
# Therefore ALL 63 fields may safely be checked
# for unnecessary ".0".
#
# Examples:
#
#     62161.0 -> 62161
#     150.0   -> 150
#     0.0     -> 0
#     99999.0 -> 99999
#
#
# IMPORTANT:
#
# This global .0 check is safe ONLY for WHQ_G
# because we proved there are no fractional fields.
#
# DO NOT apply this rule globally to Dietary files,
# sampling weights, or genuine Lab/Exam measurements.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = (
        raw_csv_df[col]
    )


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected .0 formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary .0 formatting "
    "in any WHQ_G field."
)


# ============================================================
# 39. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 40. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 62161
    and
    final_seqn.max() == 71916
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains .0."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no .0."
)


# ============================================================
# 41. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "WHQ_G CONVERSION COMPLETE"
)


print(
    "=" * 80
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN minimum:",
    df["SEQN"].min()
)


print(
    "SEQN maximum:",
    df["SEQN"].max()
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Variables containing tiny artifacts:",
    1
)


print(
    "WHD220 zero restorations:",
    correction_count
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "Whole-number variables:",
    len(INTEGER_COLUMNS)
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary .0: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2011-2012 WHQ_G XPT -> CSV
Rows: 6,175
Columns: 63
PASS: Structure = 6,175 rows x 63 columns.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 97

Tiny values by variable:
WHD220    97

PASS: Exact WHD220 97-value tiny-zero pattern validated.

--- VALIDATED ZERO RESTORATION ---
WHD220 restored to zero: 97
PASS: 97 validated WHD220 zeros restored.

Tiny values remaining after correction: 0
PASS: No tiny XPORT artifacts remain.

--- GENUINE DECIMAL CHECK ---
PASS: No genuine fractional variables.

--- SEQN ---
Minimum: 62161
Maximum: 71916
Missing: 0
Duplicates: 0
PASS: SEQN validated.

PASS: All 63 WHQ_G fields converted to nullable Int64 without rounding.
PASS: WHD010 matches CDC.
PASS: WHD020 matches CDC.
PASS: WHQ030 matches CDC.
PASS: WHQ040 matches CDC.
PASS: WHD045 matches CDC.
PASS: WHD050 matches CDC.
PASS: WHQ060 matches CDC.
PASS: WHQ070 matches CDC.
PASS: WHQ090 matches CDC.
PASS: WHQ210 matches CDC.

--- WHD220 ---
Res